# Exercise 2 — max_drawdown

Drawdown measures how far a portfolio fell from its peak. Maximum drawdown is the worst peak-to-trough decline over the entire history — the single number every investor asks about before allocating capital. A drawdown of -0.30 means the strategy fell 30% from its prior high.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def compute_returns(df):
    return df["Close"].pct_change()

def compute_equity(returns, initial=1.0):
    return (1 + returns.fillna(0)).cumprod() * initial

def max_drawdown(equity):
    """Maximum peak-to-trough drawdown.

    Returns a float ≤ 0.
    A return of 0.0 means equity never fell below its starting point.
    -0.25 means the worst decline was 25% from the prior peak.

    Steps:
      1. peak     = equity.cummax()          — running maximum up to each point
      2. drawdown = (equity - peak) / peak   — fractional distance below peak
      3. return   float(drawdown.min())      — deepest trough
    """
    # TODO: implement the 3 steps above
    return 0.0


### Checks

In [ ]:
import warnings
checks = 0

# 1 — returns a float
try:
    df = _synthetic()
    eq = compute_equity(compute_returns(df))
    dd = max_drawdown(eq)
    assert isinstance(dd, float), f"expected float, got {type(dd)}"
    checks += 1; print("✅ 1 max_drawdown returns a float")
except Exception as e:
    print("❌ 1:", e)

# 2 — drawdown is always ≤ 0
try:
    df = _synthetic()
    eq = compute_equity(compute_returns(df))
    dd = max_drawdown(eq)
    assert dd <= 1e-9, f"drawdown should be ≤ 0, got {dd}"
    checks += 1; print("✅ 2 max_drawdown is ≤ 0")
except Exception as e:
    print("❌ 2:", e)

# 3 — monotonically rising equity → drawdown ≈ 0
try:
    rising = pd.Series([1.0 + 0.01 * i for i in range(100)])
    dd = max_drawdown(rising)
    assert abs(dd) < 1e-9, f"monotone rise: expected 0, got {dd}"
    checks += 1; print("✅ 3 monotonically rising equity → drawdown ≈ 0")
except Exception as e:
    print("❌ 3:", e)

# 4 — known 50 % drawdown: 1 → 2 → 1
try:
    dates = pd.date_range("2023-01-01", periods=5, freq="B")
    test_eq = pd.Series([1.0, 1.5, 2.0, 1.5, 1.0], index=dates)
    dd = max_drawdown(test_eq)
    assert abs(dd - (-0.5)) < 1e-9, f"expected -0.5, got {dd}"
    checks += 1; print("✅ 4 equity 1→2→1 gives max_drawdown = -0.5")
except Exception as e:
    print("❌ 4:", e)

# 5 — deeper trough dominates a shallower one
try:
    dates = pd.date_range("2023-01-01", periods=7, freq="B")
    eq = pd.Series([1.0, 0.9, 1.1, 0.7, 0.8, 1.2, 1.3], index=dates)
    # Peak before index 3 = 1.1 → trough 0.7 → dd = (0.7-1.1)/1.1 ≈ -0.364
    dd = max_drawdown(eq)
    assert dd < -0.35, f"expected dd < -0.35, got {dd}"
    checks += 1; print("✅ 5 deepest trough dominates shallower ones")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
